In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from jupyter_dash import JupyterDash as Dash 
from dash import dcc, html

# --- External Stylesheet for Aesthetic Design ---
EXTERNAL_STYLESHEET = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = Dash(__name__, external_stylesheets=EXTERNAL_STYLESHEET)

# ==============================================================================
# 1. DATA LOADING AND PREPARATION
# ==============================================================================

# --- A. Metrics Data ---
try:
    df_metrics = pd.read_csv("model_metrics.csv")
    df_metrics = df_metrics.drop(columns=[df_metrics.columns[0]])
    df_melted = df_metrics.melt(id_vars='Model', var_name='Metric', value_name='Value')
except FileNotFoundError:
    print("Warning: 'model_metrics.csv' not found. Model Evaluation figures may use dummy data.")
    df_melted = pd.DataFrame({'Model': ['Lasso Regression', 'XGBoost', 'Lasso Regression', 'XGBoost'],
                              'Metric': ['MAE (minutes)', 'MAE (minutes)', 'RMSE (minutes)', 'RMSE (minutes)'],
                              'Value': [0.757, 2.053, 0.871, 2.368]})

# --- B. Predictions Data ---
try:
    df_pred = pd.read_csv("model_predictions.csv")
except FileNotFoundError:
    print("Warning: 'model_predictions.csv' not found. Model Comparison figures may use dummy data.")
    df_pred = pd.DataFrame({'Actual_Trip_Time': [13.6, 16.2, 7.62], 'Predicted_Trip_Time': [14.15, 16.57, 6.26], 'Model': ['Lasso Regression', 'Lasso Regression', 'Lasso Regression']})

# --- C. Input Data ---
try:
    df_input = pd.read_csv("ride_dataset.csv")
    day_order = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
    # Create user-friendly time string
    df_input['time_of_day'] = (df_input['pickup_minute_of_day'] // 60).astype(int).astype(str).str.zfill(2) + ':' + \
                              (df_input['pickup_minute_of_day'] % 60).astype(int).astype(str).str.zfill(2)
except FileNotFoundError:
    print("Warning: 'ride_dataset.csv' not found. Data Analysis figures will not load.")
    df_input = pd.DataFrame({'day_of_week':[], 'trip_time_minutes':[], 'trip_distance_units':[], 'pickup_minute_of_day':[], 'zone_speed_limit':[], 'wait_time_minutes':[], 'time_of_day':[]})


# ==============================================================================
# 2. FIGURE GENERATION (5 Figures Total)
# ==============================================================================

def create_card_div(figure, chart_title, insight_text):
    """Helper function to wrap a Plotly figure in the aesthetic Dash DIV structure."""
    return html.Div(
        style={'padding': '15px 0', 'marginBottom': '30px', 'backgroundColor': 'white', 'borderRadius': '8px', 'boxShadow': '0 2px 4px 0 rgba(0,0,0,0.1)'},
        children=[
            dcc.Graph(figure=figure),
            html.P(children=[
                html.B(f'{chart_title} Insight: '),
                insight_text
            ], style={'textAlign': 'center', 'fontSize': '0.9em', 'marginTop': '10px', 'paddingBottom': '10px'})
        ]
    )

# --- FIGURE 1: Bar Chart (Model Metrics) ---
fig_bar = px.bar(
    df_melted, x='Model', y='Value', color='Metric', barmode='group',
    title='1. Model Accuracy: MAE and RMSE', labels={'Value': 'Error Value (minutes)'}
)
fig_bar.update_layout(plot_bgcolor='white', margin={'t': 40, 'b': 20, 'l': 20, 'r': 20}, title_font_size=16)


# --- FIGURE 2: Scatter Plot (Actual vs. Predicted) ---
max_val = max(df_pred['Actual_Trip_Time'].max(), df_pred['Predicted_Trip_Time'].max()) if not df_pred.empty else 20
min_val = min(df_pred['Actual_Trip_Time'].min(), df_pred['Predicted_Trip_Time'].min()) if not df_pred.empty else 0
padding = 1
perfect_prediction_line = np.linspace(min_val - padding, max_val + padding, 100)

fig_scatter_pred = px.scatter(
    df_pred, x='Actual_Trip_Time', y='Predicted_Trip_Time', color='Model',
    title='2. Model Performance: Actual vs. Predicted Trip Time',
    labels={'Actual_Trip_Time': 'Actual Trip Time (minutes)', 'Predicted_Trip_Time': 'Predicted Trip Time (minutes)'},
    color_discrete_map={'Lasso Regression': 'blue', 'XGBoost': 'red'}
)
fig_scatter_pred.add_trace(go.Scatter(x=perfect_prediction_line, y=perfect_prediction_line, mode='lines', name='Perfect Prediction (y=x)', line=dict(color='gray', dash='dash'), hoverinfo='none'))
fig_scatter_pred.update_layout(plot_bgcolor='white', margin={'t': 40, 'b': 20, 'l': 20, 'r': 20}, title_font_size=16, yaxis=dict(scaleanchor="x", scaleratio=1))


# --- FIGURE 3: Scatter Plot - Distance vs. Time ---
fig_scatter_dist = px.scatter(
    df_input, x='trip_distance_units', y='trip_time_minutes', title='3. Relationship: Trip Distance vs. Trip Time',
    labels={'trip_distance_units': 'Trip Distance (units)', 'trip_time_minutes': 'Trip Time (minutes)'},
    trendline='ols'
)
fig_scatter_dist.update_layout(plot_bgcolor='white', margin={'t': 40, 'b': 20, 'l': 20, 'r': 20}, title_font_size=16)


# --- FIGURE 4: Scatter Plot - Time of Day vs. Time ---
fig_scatter_time = px.scatter(
    df_input, x='pickup_minute_of_day', y='trip_time_minutes', color='zone_speed_limit', size='wait_time_minutes', 
    hover_data=['time_of_day', 'day_of_week', 'zone_speed_limit', 'wait_time_minutes'],
    title='4. Temporal Effect: Trip Time by Minute of Day',
    labels={'pickup_minute_of_day': 'Minute of Day', 'trip_time_minutes': 'Trip Time (minutes)', 'zone_speed_limit': 'Zone Speed Limit (mph)'}
)
fig_scatter_time.update_layout(plot_bgcolor='white', margin={'t': 40, 'b': 20, 'l': 20, 'r': 20}, title_font_size=16)


# --- FIGURE 5: Box Plot - Trip Time by Day of Week ---
fig_box_day = px.box(
    df_input, x='day_of_week', y='trip_time_minutes', color='day_of_week',
    category_orders={'day_of_week': day_order},
    title='5. Categorical Effect: Trip Time Distribution by Day of Week',
    labels={'day_of_week': 'Day of Week', 'trip_time_minutes': 'Trip Time (minutes)'}
)
fig_box_day.update_layout(plot_bgcolor='white', showlegend=False, margin={'t': 40, 'b': 20, 'l': 20, 'r': 20}, title_font_size=16)


# ==============================================================================
# 3. DASH LAYOUT (Divided into two main sections)
# ==============================================================================

app.layout = html.Div(
    style={'maxWidth': '900px', 'margin': '0 auto', 'padding': '20px', 'fontFamily': 'Arial, sans-serif'},
    children=[
        html.H1(children='Comprehensive Model & Data Analysis Dashboard',
                style={'textAlign': 'center', 'color': '#2C3E50', 'borderBottom': '2px solid #3498DB', 'paddingBottom': '10px', 'marginBottom': '40px'}),

        
        # ----------------------------------------------------------------------
        # SECTION 1: MODEL EVALUATION & COMPARISON
        # ----------------------------------------------------------------------
        html.H2(children='Section 1: Model Evaluation & Comparison',
                style={'textAlign': 'center', 'color': '#3498DB', 'borderBottom': '1px solid #ECF0F1', 'paddingBottom': '10px', 'marginBottom': '30px'}),

        create_card_div(
            fig_bar, 'Metrics',
            'Lasso Regression consistently shows significantly lower MAE and RMSE values, indicating superior accuracy over XGBoost.'
        ),
        
        create_card_div(
            fig_scatter_pred, 'Prediction Performance',
            'Lasso Regression\'s (blue) predictions cluster much closer to the dashed "Perfect Prediction" line ($y=x$) than XGBoost\'s (red), visually confirming its better performance.'
        ),

        
        # ----------------------------------------------------------------------
        # SECTION 2: DATA ANALYSIS
        # ----------------------------------------------------------------------
        html.H2(children='Section 2: Input Data Feature Analysis',
                style={'textAlign': 'center', 'color': '#3498DB', 'borderBottom': '1px solid #ECF0F1', 'paddingTop': '20px', 'paddingBottom': '10px', 'marginBottom': '30px', 'marginTop': '40px'}),
        
        create_card_div(
            fig_scatter_dist, 'Distance vs. Time',
            'Trip time is strongly correlated with distance, forming the foundational linear relationship that the predictive models rely on.'
        ),
        
        create_card_div(
            fig_scatter_time, 'Time of Day',
            'Higher trip times are often associated with larger circles (longer wait times) and lower speed limits (lighter colors), suggesting that traffic/congestion effects are critical features.'
        ),
        
        create_card_div(
            fig_box_day, 'Day of Week',
            'The distribution of trip times varies by the day of the week, highlighting that temporal features capture important variance in traffic patterns.'
        ),

        html.Div(style={'height': '20px'}) # Final spacer
    ]
)

# 4. Run the app in the notebook
app.run(mode='inline')

C:\Users\sonia\anaconda3\lib\site-packages\dash\dash.py:634: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.

